# PrimeKG Implementation
Will use `Llama-3.1-8B-Instruct`

In [3]:
import kuzu
import pandas as pd

In [4]:
kg = pd.read_csv("data/raw/kg.csv", low_memory=False)

x = kg[["x_index", "x_id", "x_name", "x_type", "x_source"]].rename(
    columns={"x_index": "node_index", "x_id": "id", "x_name": "name", "x_type": "type", "x_source": "source"})
y = kg[["y_index", "y_id", "y_name", "y_type", "y_source"]].rename(
    columns={"y_index": "node_index", "y_id": "id", "y_name": "name", "y_type": "type", "y_source": "source"})

entities = pd.concat([x, y]).drop_duplicates(subset="node_index").reset_index(drop=True)
entities.to_csv("data/processed/entities.csv", index=False)
print(f"{len(entities)} unique entities")

129375 unique entities


In [5]:
edges = kg[["x_index", "y_index", "relation", "display_relation"]].rename(
    columns={"x_index": "src", "y_index": "dst"})
edges.to_csv("data/processed/edges.csv", index=False)
print(f"{len(edges)} edges")

8100498 edges


In [7]:
db = kuzu.Database("../primekg_kuzu")
conn = kuzu.Connection(db)

conn.execute("""
    CREATE NODE TABLE Entity(
        node_index INT64,
        id STRING,
        name STRING,
        type STRING,
        source STRING,
        PRIMARY KEY(node_index)
    )
""")

conn.execute("""
    CREATE REL TABLE RELATES_TO(
        FROM Entity TO Entity,
        relation STRING,
        display_relation STRING
    )
""")
print("Schema created")

Schema created


In [ ]:
conn.execute("""
    COPY Entity FROM 'data/processed/entities.csv'
    (HEADER=true, DELIM=',', QUOTE='"', ESCAPE='"')
""")
conn.execute("""
    COPY RELATES_TO FROM 'data/processed/edges.csv'
    (HEADER=true, DELIM=',', QUOTE='"', ESCAPE='"')
""")
print("Data loaded")

RuntimeError: Copy exception: Error in file data/processed/entities.csv on line 14069: expected 5 values per row, but got more. Line/record containing the error: '14067,DB01838,"6,4'-Dihydroxy-3-Methyl-3',5'-Dibromoflavone",drug...'

In [ ]:
conn.execute("CALL CREATE_FTS_INDEX('Entity', 'entity_fts', ['name'])")
print("FTS index built")

In [ ]:
result = conn.execute("""
    CALL QUERY_FTS_INDEX('Entity', 'entity_fts', 'Alzheimer')
    YIELD node, score
    RETURN node.name, score
    ORDER BY score DESC
    LIMIT 5
""")
while result.has_next():
    print(result.get_next())